In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'covid_19'
SOURCE_SCHEMA_NAME = 'bronze'
SOURCE_TABLE_NAME = 'locations'

TARGET_CATALOG_NAME = 'covid_19'
TARGET_SCHEMA_NAME = 'silver'
TARGET_TABLE_NAME = 'locations'

In [0]:
df = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')
    .select(
        F.col('location').alias('country'),
        F.col('iso_code'),
        F.col('vaccines'),
        F.col('last_observation_date'),
        F.col('source_name'),
        F.col('source_website')
    )
    .withColumn(
        'vaccine_array',
        F.transform(
            F.split(F.col('vaccines'), ','),
            lambda x: F.trim(x)
        )
    )
    .withColumn('vaccine_count', F.size('vaccine_array'))
    .filter(F.col('country').isNotNull())
    .filter(F.col('iso_code').isNotNull())
)

In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')